# Dimension Mapping - Cross-System Asset Discovery

Harmonizes unit/equipment identifiers across PI, GADS, and iCare for **Riverton (RV2 & RV3)**.

**Inputs:**
- `Files/DimensionMapping/unit_mapping.csv` - unit-level cross-reference
- `Files/DimensionMapping/acronym_dictionary.csv` - editable abbreviation lookup
- Delta tables: `pi_tags_metadata`, `gads_unit_configuration`, `gads_unit_event`, `icare_assets`, `icare_analyses`

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

## 1. Load mapping tables

In [ ]:
# Unit mapping
df_unit_map = (spark.read.option("header", True).option("inferSchema", True)
    .csv("Files/DimensionMapping/unit_mapping.csv")
)
df_unit_map.write.mode("overwrite").format("delta").saveAsTable("dim_unit_mapping")
print("dim_unit_mapping:")
df_unit_map.show(truncate=False)

# Acronym dictionary
df_acronyms = (spark.read.option("header", True).option("inferSchema", True)
    .csv("Files/DimensionMapping/acronym_dictionary.csv")
)
df_acronyms.write.mode("overwrite").format("delta").saveAsTable("dim_acronym_dictionary")
print(f"\ndim_acronym_dictionary: {df_acronyms.count()} entries")
df_acronyms.show(truncate=False)

## 2. PI Tags - RV2 & RV3 only

In [ ]:
df_pi = spark.table("pi_tags_metadata").filter(F.col("Plant").isin("RV2", "RV3"))

# Extract equipment group with multiple fallback patterns
# Pattern 1: RV2:BATU2BT05.AG -> BAT (prefix before U2/U3)
# Pattern 2: RV2:BFPP02A.AG  -> BFP (leading letters before digit)
# Pattern 3: RV2:T2GEN01.AG  -> T (single letter equipment code) - skip
df_pi = (df_pi
    .withColumn("tag_body", F.regexp_extract("Tag", r"^LG\d:([^.]+)", 1))
    .withColumn("equip_prefix_v1", F.regexp_extract("tag_body", r"^([A-Z]+?)U[23]", 1))
    .withColumn("equip_prefix_v2", F.regexp_extract("tag_body", r"^([A-Z]{2,})\d", 1))
    .withColumn("equip_prefix_v3", F.regexp_extract("tag_body", r"^([A-Z]{2,})", 1))
    .withColumn("equip_prefix",
        F.when(F.length("equip_prefix_v1") > 0, F.col("equip_prefix_v1"))
         .when(F.length("equip_prefix_v2") > 0, F.col("equip_prefix_v2"))
         .otherwise(F.col("equip_prefix_v3"))
    )
    .drop("equip_prefix_v1", "equip_prefix_v2", "equip_prefix_v3")
)

# Cache because we'll reference repeatedly
df_pi.cache()

prefix_coverage = df_pi.filter(F.length("equip_prefix") == 0).count()
print(f"PI tags in scope: {df_pi.count()}")
print(f"PI tags with no extractable prefix: {prefix_coverage}")

print("\nPI equipment groups by prefix:")
df_pi.groupBy("Plant", "equip_prefix").agg(
    F.count("*").alias("tag_count"),
    F.first("Descriptor").alias("sample_descriptor")
).orderBy("Plant", F.desc("tag_count")).show(60, truncate=60)

## 3. GADS - Riverton units

In [ ]:
# Filter GADS to LG units
lg_gads_ids = [86, 87]  # RVTONG2, RVTONG3

df_gads_units = spark.table("gads_unit_configuration").filter(F.col("UNIT_ID").isin(lg_gads_ids))
print("GADS Unit Configuration - Riverton:")
df_gads_units.select("UNIT_ID", "OTS_UNIT_ID", "UNIT_TYPE", "MAX_DESIGN_CAP", "PRIMARY_FUEL",
                     "BOILER_TYPE", "BOILER_MANF", "TURBINE_TYPE", "TURBINE_MANF").show(truncate=False)

# GADS events for these units
df_gads_events = spark.table("gads_unit_event").filter(F.col("UNIT_ID").isin(lg_gads_ids))
print(f"GADS events for LG units: {df_gads_events.count()}")

# GADS equipment type outage - structured equipment list per unit
df_equip_outage = spark.table("gads_equip_type_outage").filter(F.col("UNIT_ID").isin(lg_gads_ids))
print(f"\nGADS equipment type outage records for LG: {df_equip_outage.count()}")
print("\nEquipment types tracked per unit:")
df_equip_outage.select("UNIT_ID", "EQUIP_TYPE", "LAST_OUTAGE_DT").orderBy("UNIT_ID", "EQUIP_TYPE").show(60, truncate=60)

# Full equipment type catalog
df_equip_type = spark.table("gads_equip_type")
print(f"\nGADS master equipment type catalog: {df_equip_type.count()} entries")

## 4. iCare - Riverton asset hierarchy

In [ ]:
LG_ICARE_PLANT_ID = "5e14fa24feb9a596df5ca55b"

df_assets = spark.table("icare_assets")
all_asset_rows = df_assets.select("_id", "name", "ParentId").collect()

# Build parent -> children index in driver memory (small enough)
by_id = {r["_id"]: (r["name"], r["ParentId"]) for r in all_asset_rows}
children_of = {}
for aid, (_n, pid) in by_id.items():
    children_of.setdefault(pid, []).append(aid)

# BFS walk under Riverton with depth tracking
level_of = {LG_ICARE_PLANT_ID: 0}
queue = [LG_ICARE_PLANT_ID]
while queue:
    nxt = []
    for pid in queue:
        for cid in children_of.get(pid, []):
            if cid not in level_of:
                level_of[cid] = level_of[pid] + 1
                nxt.append(cid)
    queue = nxt

# Build descendant set per node (subtree membership) for unit attribution
def collect_subtree(root):
    out = set()
    stack = [root]
    while stack:
        n = stack.pop()
        out.add(n)
        stack.extend(children_of.get(n, []))
    return out

# Walk up to find root unit for any equipment via parent chain
def ancestors(node_id):
    chain = []
    cur = node_id
    while cur and cur in by_id:
        chain.append(cur)
        cur = by_id[cur][1]
        if len(chain) > 20:  # safety
            break
    return chain

lg_ids = set(level_of.keys())
print(f"iCare hierarchy under Riverton:")
for lvl in sorted(set(level_of.values())):
    nodes = [aid for aid, l in level_of.items() if l == lvl]
    print(f"  Level {lvl}: {len(nodes)} nodes")

# Levels 1 / 2 / 3 helpers
level1_ids = [aid for aid, l in level_of.items() if l == 1]
level2_ids = [aid for aid, l in level_of.items() if l == 2]
level3_ids = [aid for aid, l in level_of.items() if l == 3]
deep_ids   = [aid for aid, l in level_of.items() if l >= 4]

print(f"\n  Level 1 (functional areas): {len(level1_ids)}")
print(f"  Level 2 (units/groups):     {len(level2_ids)}")
print(f"  Level 3 (equipment):        {len(level3_ids)}")
print(f"  Level 4+ (sub-components):  {len(deep_ids)}")

level1 = df_assets.filter(F.col("_id").isin(level1_ids)).select("_id", "name", "ParentId")
level2 = df_assets.filter(F.col("_id").isin(level2_ids)).select("_id", "name", "ParentId")
level3 = df_assets.filter(F.col("_id").isin(level3_ids)).select("_id", "name", "ParentId")
df_assets_deep = df_assets.filter(F.col("_id").isin(deep_ids)).select("_id", "name", "ParentId")

print("\nLevel 1 - Functional Areas:")
level1.show(truncate=80)
print("Level 2 - Units/Groups:")
level2.show(truncate=80)
print("Level 3 - Equipment (sample):")
level3.show(20, truncate=80)
if deep_ids:
    print("Level 4+ - Sub-components (sample):")
    df_assets_deep.show(15, truncate=80)

# Collect all equipment rows for cross-system matching
all_equip = level2.union(level3).union(df_assets_deep)
icare_equip_rows = all_equip.collect()
print(f'iCare equipment rows collected: {len(icare_equip_rows)}')

# Unit attribution for each iCare equipment node
icare_equip_unit = {}
for row in icare_equip_rows:
    eid = row['_id']
    ename = (row['name'] or '').lower()
    nm = ' ' + ename + ' '
    if any(p in nm for p in [' u2 ', ' rv2 ', ' unit 2 ', ' #2 ']):
        icare_equip_unit[eid] = 'RV2'
    elif any(p in nm for p in [' u3 ', ' rv3 ', ' unit 3 ', ' #3 ']):
        icare_equip_unit[eid] = 'RV3'
    else:
        icare_equip_unit[eid] = None
print(f'Unit attribution: {sum(1 for v in icare_equip_unit.values() if v)} of {len(icare_equip_unit)} attributed')


## 5. AIMM Entity Hierarchy - The Base Spine

AIMM is the maintenance management system (AIM database). Its entity hierarchy
has 284K entities across 64 sites with 6 levels of depth and 84 entity types.
We use it as the **base dimension** because:
- Direct link to work requests (ENTITY_IDENTITY = mtnoi)
- Granular typing (Valve, Pump, Motor, Turbine, etc.)
- BOM numbers for parts tracking
- Actively maintained by plant staff
- Covers all Contoso plants (scalable)



In [ ]:
# Load AIMM hierarchy from Eventhouse (pi-realtime-db)
# Uses the Fabric-native Kusto Spark connector (pre-installed)

kusto_cluster = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"
kusto_db = "pi-realtime-db"

# Option 1: Read via Kusto Spark connector
df_aimm_raw = (spark.read
    .format("com.microsoft.kusto.spark.synapse.datasource")
    .option("spark.synapse.linkedService", "pi-realtime-eventhouse")
    .option("kustoCluster", kusto_cluster)
    .option("kustoDatabase", kusto_db)
    .option("kustoQuery", "AimmEntityHierarchyRaw | project-away source_host")
    .load()
)

# Fallback: If connector above fails, read via notebookutils + REST
# from notebookutils import mssparkutils
# Uncomment below if the Kusto connector is not available:
# df_aimm_raw = spark.read.format("delta").table("aimm_entity_hierarchy")  # if synced to lakehouse

# Riverton site constants
LG_SITE_OI = 7000000
LG_U2_MTNOI = 7000002
LG_U3_MTNOI = 7000003
LG_COMMON_MTNOI = 7000004

df_aimm = df_aimm_raw.filter(F.col("siteoi") == LG_SITE_OI)
df_aimm.cache()

print(f"AIMM hierarchy total: {df_aimm_raw.count()} entities")
print(f"AIMM Riverton: {df_aimm.count()} entities")
print(f"\nEntity types:")
df_aimm.groupBy("entity_type").agg(F.count("*").alias("count")).orderBy(F.desc("count")).show(30, truncate=False)


In [ ]:
# Build the AIMM hierarchy structure
# The level columns represent the ANCESTOR LINEAGE:
#   level1_mtnoi = site root (7000000)
#   level2_mtnoi = unit (7000002=U2, 7000003=U3, 7000004=Common)
#   level3_mtnoi = process/subprocess (e.g., Turbine U3, Feedwater U3)
#   level4_mtnoi = system within subprocess
#   level5_mtnoi = equipment
#   level6_mtnoi = sub-component

# Determine hierarchy level for each entity
df_aimm_leveled = (df_aimm
    .withColumn("hier_level", 
        F.when(F.col("level2_mtnoi").isNull(), 1)  # Site root or Unit
         .when(F.col("level3_mtnoi").isNull(), 2)  # Direct child of unit (process/subprocess)
         .when(F.col("level4_mtnoi").isNull(), 3)  # System
         .when(F.col("level5_mtnoi").isNull(), 4)  # Equipment
         .when(F.col("level6_mtnoi").isNull(), 5)  # Sub-component
         .otherwise(6)
    )
    .withColumn("plant",
        F.when(F.col("level2_mtnoi") == LG_U2_MTNOI, F.lit("RV2"))
         .when(F.col("level2_mtnoi") == LG_U3_MTNOI, F.lit("RV3"))
         .when(F.col("level2_mtnoi") == LG_COMMON_MTNOI, F.lit("Common"))
         .when(F.col("mtnoi") == LG_U2_MTNOI, F.lit("RV2"))
         .when(F.col("mtnoi") == LG_U3_MTNOI, F.lit("RV3"))
         .when(F.col("mtnoi") == LG_COMMON_MTNOI, F.lit("Common"))
         .otherwise(F.lit(None))
    )
)

print("AIMM hierarchy by level:")
df_aimm_leveled.groupBy("hier_level").agg(
    F.count("*").alias("entities"),
    F.countDistinct("entity_type").alias("types")
).orderBy("hier_level").show()

print("\nAIMM hierarchy by plant:")
df_aimm_leveled.groupBy("plant").agg(
    F.count("*").alias("entities")
).orderBy("plant").show()

# Persist as Delta table for downstream use
df_aimm_leveled.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("dim_aimm_hierarchy")
print(f"Persisted dim_aimm_hierarchy: {df_aimm_leveled.count()} rows")

# 


In [ ]:
# --- Shared utilities: tokenizer + confidence tiers ---
# (Used by PI->AIMM and iCare->AIMM matching below)
import re

# Acronym dictionaries from dim_acronym_dictionary
acronym_to_words = {}  # 'bfp' -> {'boiler','feed','pump'}
phrase_to_acronym = {} # 'boiler feed pump' -> 'BFP'

for r in df_acronyms.collect():
    abbr = r["abbreviation"].strip()
    exp = r["expansion"].strip()
    if not abbr or not exp: continue
    words = re.findall(r"[a-z]+", exp.lower())
    acronym_to_words[abbr.lower()] = set(words)
    phrase_to_acronym[exp.lower()] = abbr

noise = {"a","b","c","d","e","w","n","s","the","to","from","of","in",
         "at","and","or","on","no","is","it","an","by","up","do",
         "unit","little","riverton","lg","ir","window","bucket"}

def tokenize_full(text):
    """Tokenize with bidirectional acronym expansion."""
    text = (text or "").lower()
    words = re.findall(r"[a-z]+", text)
    tokens = set(words)
    for w in list(words):
        if w in acronym_to_words:
            tokens |= acronym_to_words[w]
    full = " ".join(words)
    for phrase, abbr in phrase_to_acronym.items():
        if " " in phrase and phrase in full:
            tokens.add(abbr.lower())
    return tokens - noise

def confidence_tier(jaccard, shared_count):
    if jaccard >= 0.50 or shared_count >= 7: return "high"
    if jaccard >= 0.30 or shared_count >= 5: return "medium"
    if jaccard >= 0.20 or shared_count >= 4: return "low"
    return None

# PI groups (pre-computed for matching)
df_pi_summary = (df_pi.select("Plant", "Tag", "Descriptor", "EngineeringUnits", "equip_prefix")
    .withColumn("descriptor_lower", F.lower("Descriptor")))
pi_groups = df_pi_summary.groupBy("Plant", "equip_prefix").agg(
    F.collect_set("descriptor_lower").alias("descriptors"),
    F.collect_set(F.lower("EngineeringUnits")).alias("engunits"),
    F.count("*").alias("tag_count")
).collect()

print(f"Tokenizer ready: {len(acronym_to_words)} acronyms, {len(pi_groups)} PI prefix groups")

## 6. Map PI Tags -> AIMM Entities
PI tag prefixes (BFP, TBY, CWT, etc.) map naturally to AIMM subprocess descriptions:
- `BFP` -> "Feedwater U3-Boilerfeed Pump/Turb Sys"
- `TBY` -> "Turbine U3-High,Intermediate,Low Pressure Sections"
- `CWT` -> "Cooling Water U3-..."
- `GEN` -> "Generator U3-..."
We use the acronym dictionary + AIMM entity descriptions for matching.



In [ ]:
# Map PI tag prefixes to AIMM entities (EXPANDED matching)
# Previously: only matched against Subprocess/Process/System entity types (~120)
# Now: matches against ALL entities at levels 2-5 (~3200+) including
# equipment-level entities (Valve, Pump, Motor, Fan, Generator, etc.)
# Also adds direct prefix-to-entity_id fallback for remaining unmapped groups

import re

# --- Phase 0: Collect AIMM entities at levels 2-5 for matching ---
aimm_match_entities = df_aimm_leveled.filter(
    (F.col("hier_level").between(2, 5)) & 
    (F.col("plant").isin("RV2", "RV3"))
).select("mtnoi", "entity_id", "entity_descr", "entity_type", "plant", 
         "hier_level", "level2_mtnoi", "level3_mtnoi").collect()

print(f"AIMM entities for matching: {len(aimm_match_entities)} (was ~120 with type filter)")

# Token index: include entity_type for richer matching
# e.g. "Pump" entity type adds "pump" token, helping match PI pump tags
aimm_tokens = {}
for row in aimm_match_entities:
    eid = row["mtnoi"]
    descr = (row["entity_descr"] or "").lower()
    entity_id = (row["entity_id"] or "").lower()
    etype = (row["entity_type"] or "").lower()
    combined = descr + " " + entity_id + " " + etype
    tokens = tokenize_full(combined)
    aimm_tokens[eid] = (row["entity_descr"], row["plant"], row["hier_level"], tokens, entity_id)

# --- Phase 1: Jaccard fuzzy matching (wider entity pool) ---
aimm_matches = []
for row in pi_groups:
    plant = row["Plant"]
    prefix = row["equip_prefix"]
    descriptors = row["descriptors"] or []
    engunits = row["engunits"] or []
    
    pi_text = " ".join(descriptors) + " " + " ".join(filter(None, engunits))
    if prefix:
        pi_text = prefix + " " + pi_text
    pi_tokens = tokenize_full(pi_text)
    if not pi_tokens:
        continue
    
    for eid, (ename, aimm_plant, aimm_level, etokens, _eid_str) in aimm_tokens.items():
        if aimm_plant and aimm_plant != plant:
            continue
        if not etokens:
            continue
        shared = pi_tokens & etokens
        if not shared:
            continue
        union = pi_tokens | etokens
        jaccard = len(shared) / len(union) if union else 0.0
        tier = confidence_tier(jaccard, len(shared))
        if not tier:
            continue
        aimm_matches.append((plant, prefix, ", ".join(sorted(descriptors)[:3]),
                            eid, ename, aimm_plant or "shared", aimm_level,
                            len(shared), round(jaccard, 3), tier,
                            ", ".join(sorted(shared))))

# --- Phase 2: Direct prefix fallback for still-unmapped groups ---
# PI prefix "GEN" matches AIMM entity_id starting with "gen"
fuzzy_prefixes = set((m[0], m[1]) for m in aimm_matches)
for row in pi_groups:
    plant = row["Plant"]
    prefix = row["equip_prefix"]
    if not prefix or len(prefix) < 3 or (plant, prefix) in fuzzy_prefixes:
        continue
    descriptors = row["descriptors"] or []
    pfx = prefix.lower()
    for eid, (ename, aimm_plant, aimm_level, etokens, eid_str) in aimm_tokens.items():
        if aimm_plant and aimm_plant != plant:
            continue
        if eid_str.startswith(pfx) or pfx in eid_str.replace("-", " ").split():
            aimm_matches.append((plant, prefix, ", ".join(sorted(descriptors)[:3]),
                                eid, ename, aimm_plant or "shared", aimm_level,
                                1, 0.15, "low", f"prefix:{pfx}"))
            fuzzy_prefixes.add((plant, prefix))

if aimm_matches:
    match_schema = StructType([
        StructField("pi_plant", StringType()),
        StructField("pi_prefix", StringType()),
        StructField("pi_descriptors", StringType()),
        StructField("aimm_mtnoi", IntegerType()),
        StructField("aimm_descr", StringType()),
        StructField("aimm_plant", StringType()),
        StructField("aimm_level", IntegerType()),
        StructField("shared_token_count", IntegerType()),
        StructField("jaccard", DoubleType()),
        StructField("confidence", StringType()),
        StructField("shared_keywords", StringType()),
    ])
    df_pi_aimm = spark.createDataFrame(aimm_matches, match_schema)
    
    from pyspark.sql.window import Window
    # Rank: best Jaccard, most shared tokens, prefer higher-level entity as tiebreaker
    w = Window.partitionBy("pi_plant", "pi_prefix").orderBy(
        F.desc("jaccard"), F.desc("shared_token_count"), F.asc("aimm_level"))
    df_pi_aimm_ranked = df_pi_aimm.withColumn("rank", F.row_number().over(w))
    
    print(f"PI->AIMM matches: {df_pi_aimm.count()} total")
    matched_count = df_pi_aimm_ranked.filter(F.col("rank") == 1).select("pi_plant", "pi_prefix").distinct().count()
    total_count = len(pi_groups)
    print(f"Unique prefix groups matched: {matched_count} / {total_count}")
    
    print("\nTop match per PI prefix:")
    df_pi_aimm_ranked.filter(F.col("rank") == 1).select(
        "pi_plant", "pi_prefix", "confidence", "jaccard", "aimm_level", "aimm_descr", "shared_keywords"
    ).orderBy("pi_plant", "pi_prefix").show(80, truncate=70)
    
    df_pi_aimm_ranked.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("dim_pi_to_aimm_map")
    print(f"\nPersisted dim_pi_to_aimm_map: {df_pi_aimm_ranked.count()} rows")

#

## 7. Map iCare -> AIMM Entities
Cross-reference iCare measurement points to AIMM entities so condition
monitoring data (vibration, oil analysis, thermography) links to the
maintenance hierarchy.



In [ ]:
# Map iCare equipment nodes to AIMM entities via fuzzy name matching

icare_to_aimm = []
for row in icare_equip_rows:
    icare_id = row["_id"]
    icare_name = (row["name"] or "").lower()
    icare_tokens_set = tokenize_full(icare_name)
    icare_unit = icare_equip_unit.get(icare_id)
    
    if not icare_tokens_set:
        continue
    
    best_match = None
    best_jaccard = 0
    
    for eid, (ename, aimm_plant, _level, etokens, _eid_str) in aimm_tokens.items():
        if aimm_plant and icare_unit and aimm_plant != icare_unit:
            continue
        if not etokens:
            continue
        shared = icare_tokens_set & etokens
        if not shared:
            continue
        union = icare_tokens_set | etokens
        jaccard = len(shared) / len(union) if union else 0.0
        if jaccard > best_jaccard:
            best_jaccard = jaccard
            best_match = (eid, ename, aimm_plant, len(shared), jaccard)
    
    if best_match and best_jaccard >= 0.2:
        tier = confidence_tier(best_jaccard, best_match[3])
        if tier:
            icare_to_aimm.append((icare_id, row["name"], icare_unit,
                                  best_match[0], best_match[1], best_match[2],
                                  best_match[3], round(best_match[4], 3), tier))

if icare_to_aimm:
    icare_aimm_schema = StructType([
        StructField("icare_id", StringType()),
        StructField("icare_name", StringType()),
        StructField("icare_unit", StringType()),
        StructField("aimm_mtnoi", IntegerType()),
        StructField("aimm_descr", StringType()),
        StructField("aimm_plant", StringType()),
        StructField("shared_token_count", IntegerType()),
        StructField("jaccard", DoubleType()),
        StructField("confidence", StringType()),
    ])
    df_icare_aimm = spark.createDataFrame(icare_to_aimm, icare_aimm_schema)
    df_icare_aimm.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("dim_icare_to_aimm_map")
    print(f"iCare->AIMM mappings: {df_icare_aimm.count()} rows")
    print("\nBy confidence:")
    df_icare_aimm.groupBy("confidence").agg(F.count("*").alias("count"), F.avg("jaccard").alias("avg_jaccard")).show()

# 


## 8. Final `dim_equipment - AIMM-Based Unified Dimension
One row per AIMM entity under Riverton, enriched with:
- PI tag coverage (direct + rollup)
- iCare measurement point linkage
- GADS event references
- Work request counts
- Full hierarchy path



In [ ]:
# Build final dim_equipment matching gold schema exactly
# Target: dim_equipment (same columns as semantic model)

# --- Lookup: AIMM mtnoi -> ancestor names for level0-5 ---
aimm_lookup = {r['mtnoi']: r for r in df_aimm_leveled.collect()}

def get_ancestor_name(mtnoi):
    r = aimm_lookup.get(mtnoi)
    return r['entity_descr'].strip() if r else None

# --- PI mapping lookup ---
pi_map = {}
try:
    for r in spark.table('dim_pi_to_aimm_map').filter(F.col('rank') == 1).collect():
        pi_map.setdefault(r['aimm_mtnoi'], []).append(r)
except: pass

# Actual tag counts per prefix group (for accurate pi_tag_count_direct)
prefix_tag_count = {(pg['Plant'], pg['equip_prefix']): int(pg['tag_count']) for pg in pi_groups}

# --- iCare mapping lookup ---
icare_map = {}
try:
    for r in spark.table('dim_icare_to_aimm_map').collect():
        icare_map.setdefault(r['aimm_mtnoi'], []).append(r['icare_id'])
except: pass

# --- Build rows matching dim_equipment schema ---
equip_rows = []
for row in df_aimm_leveled.filter(F.col('plant').isNotNull()).collect():
    mtnoi = row['mtnoi']
    
    # Ancestor names for level0-5
    level_names = [
        'Riverton',  # level0 = site
        get_ancestor_name(row['level2_mtnoi']),  # level1 = unit
        get_ancestor_name(row['level3_mtnoi']),  # level2 = process
        get_ancestor_name(row['level4_mtnoi']),  # level3 = system
        get_ancestor_name(row['level5_mtnoi']),  # level4 = equipment
        row['entity_descr'] if row['hier_level'] >= 5 else None  # level5 = self if deep
    ]
    full_path = ' > '.join([n for n in level_names if n])
    
    # PI coverage (count actual tags, not just prefix groups)
    pi_info = pi_map.get(mtnoi, [])
    pi_prefixes = ', '.join(sorted(set(r['pi_prefix'] for r in pi_info))) if pi_info else None
    pi_count = sum(prefix_tag_count.get((r['pi_plant'], r['pi_prefix']), 0) for r in pi_info)
    best_conf = pi_info[0]['confidence'] if pi_info else None
    best_jacc = float(pi_info[0]['jaccard']) if pi_info else None
    
    # iCare linkage
    icare_ids = icare_map.get(mtnoi, [])
    
    # Asset ID mapping (for predictions/watchlist compatibility)
    asset_id = None
    if row['plant'] == 'RV3' and row['entity_descr'] and 'Turbine' in row['entity_descr']:
        asset_id = 'RV3_U3_Steam_Turbine'
    elif row['plant'] == 'RV3' and row['entity_descr'] and 'BFP' in row['entity_descr']:
        asset_id = 'RV3_U3_Boiler_Feed_Pump_East'
    elif row['plant'] == 'RV2' and row['entity_descr'] and 'Boiler' in row['entity_descr']:
        asset_id = 'RV2_U2_Boiler'
    
    equip_rows.append((
        str(mtnoi),           # icare_id (repurposed as entity key)
        str(row['level3_mtnoi'] or row['level2_mtnoi'] or ''),  # parent
        row['entity_descr'],  # equipment_name
        int(row['hier_level']),
        row['plant'],
        True if row['hier_level'] >= 4 else False,  # is_leaf
        level_names[0], level_names[1], level_names[2],
        level_names[3], level_names[4], level_names[5],
        full_path,
        0,  # child_count (would need subtree calc)
        0,  # descendant_count
        row['plant'] if pi_info else None,  # pi_plants_direct
        pi_prefixes,
        pi_prefixes,  # rollup same as direct for now
        pi_count,
        pi_count,
        None,  # gads_acronyms_direct
        None,  # gads_acronyms_rollup
        0,     # gads_event_count_direct
        0,     # gads_event_count_rollup
        best_conf,
        best_jacc,
        asset_id,
        row['entity_descr'],  # asset_display_name
        row['plant'],         # unit
    ))

schema = StructType([
    StructField('icare_id', StringType()),
    StructField('parent_icare_id', StringType()),
    StructField('equipment_name', StringType()),
    StructField('level', IntegerType()),
    StructField('plant', StringType()),
    StructField('is_leaf', BooleanType()),
    StructField('level0_name', StringType()),
    StructField('level1_name', StringType()),
    StructField('level2_name', StringType()),
    StructField('level3_name', StringType()),
    StructField('level4_name', StringType()),
    StructField('level5_name', StringType()),
    StructField('full_path', StringType()),
    StructField('child_count', IntegerType()),
    StructField('descendant_count', IntegerType()),
    StructField('pi_plants_direct', StringType()),
    StructField('pi_prefixes_direct', StringType()),
    StructField('pi_prefixes_rollup', StringType()),
    StructField('pi_tag_count_direct', IntegerType()),
    StructField('pi_tag_count_rollup', IntegerType()),
    StructField('gads_acronyms_direct', StringType()),
    StructField('gads_acronyms_rollup', StringType()),
    StructField('gads_event_count_direct', IntegerType()),
    StructField('gads_event_count_rollup', IntegerType()),
    StructField('best_pi_match_confidence', StringType()),
    StructField('best_pi_match_jaccard', DoubleType()),
    StructField('asset_id', StringType()),
    StructField('asset_display_name', StringType()),
    StructField('unit', StringType()),
])

df_dim = spark.createDataFrame(equip_rows, schema)
df_dim.write.mode('overwrite').format('delta').option('overwriteSchema', 'true').saveAsTable('dim_equipment')

total = df_dim.count()
with_pi = df_dim.filter(F.col('pi_tag_count_direct') > 0).count()
print(f'dim_equipment: {total} rows (AIMM-based)')
print(f'  With PI mapping: {with_pi}')
print(f'  Schema matches semantic model: 29 columns')
df_dim.groupBy('plant').agg(F.count('*').alias('entities'), F.sum('pi_tag_count_direct').alias('pi_tags')).show()

In [ ]:
# Rebuild bridge_pi_tag_to_asset with AIMM entity linkage
# Each PI tag gets assigned to a specific AIMM subprocess entity
# This bridges: fact_pi[Tag] -> bridge[Tag, asset_id] -> dim_equipment[icare_id]

# Get PI->AIMM primary mappings (rank 1 per prefix)
df_pi_aimm_primary = spark.table('dim_pi_to_aimm_map').filter(F.col('rank') == 1)

# Join each PI tag to its AIMM entity via the prefix mapping
df_bridge = (df_pi
    .select('Tag', 'Plant', 'equip_prefix', 'Descriptor', 'EngineeringUnits')
    .join(
        df_pi_aimm_primary.select(
            F.col('pi_plant').alias('Plant'),
            F.col('pi_prefix').alias('equip_prefix'),
            F.col('aimm_mtnoi'),
            F.col('aimm_descr'),
            F.col('confidence')
        ),
        on=['Plant', 'equip_prefix'],
        how='left'
    )
)

# Determine tag_role and downtime_relevance based on tag characteristics
df_bridge = (df_bridge
    .withColumn('tag_role',
        F.when(F.lower('Descriptor').rlike('vibr|vib '), F.lit('health'))
         .when(F.lower('Descriptor').rlike('temp|brg|bearing'), F.lit('health'))
         .when(F.lower('Descriptor').rlike('press|flow|level'), F.lit('process'))
         .when(F.lower('Descriptor').rlike('load|mw|gen|speed'), F.lit('running_indicator'))
         .otherwise(F.lit('other'))
    )
    .withColumn('downtime_relevance',
        F.when(F.col('tag_role') == 'health', F.lit('HIGH'))
         .when(F.col('tag_role') == 'running_indicator', F.lit('HIGH'))
         .when(F.col('tag_role') == 'process', F.lit('MEDIUM'))
         .otherwise(F.lit('LOW'))
    )
)

# Output matching semantic model schema:
# Tag, tag_description, eng_units, asset_id, tag_role, downtime_relevance, notes
df_bridge_final = df_bridge.select(
    F.col('Tag'),
    F.col('Descriptor').alias('tag_description'),
    F.col('EngineeringUnits').alias('eng_units'),
    F.col('aimm_mtnoi').cast('string').alias('asset_id'),  # links to dim_equipment[icare_id]
    F.col('tag_role'),
    F.col('downtime_relevance'),
    F.concat(F.lit('AIMM: '), F.coalesce(F.col('aimm_descr'), F.lit('unmapped'))).alias('notes')
)

df_bridge_final.write.mode('overwrite').format('delta').option('overwriteSchema', 'true').saveAsTable('bridge_pi_tag_to_asset')

total = df_bridge_final.count()
mapped = df_bridge_final.filter(F.col('asset_id').isNotNull()).count()
print(f'bridge_pi_tag_to_asset: {total} tags ({mapped} mapped to AIMM, {total-mapped} unmapped)')
print(f'\\nBy role:')
df_bridge_final.groupBy('tag_role', 'downtime_relevance').agg(F.count('*').alias('tags')).orderBy('tag_role').show()
print(f'\\nSample mapped tags:')
df_bridge_final.filter(F.col('asset_id').isNotNull()).select('Tag', 'tag_description', 'asset_id', 'tag_role').show(10, truncate=60)


In [ ]:
# Analyze mapping effectiveness
from pyspark.sql import functions as F

print("=" * 60)
print("dim_equipment")
print("=" * 60)
df = spark.table("dim_equipment")
print(f"Total rows: {df.count()}")
print("\nBy plant:")
df.groupBy("plant").agg(F.count("*").alias("entities"), F.sum("pi_tag_count_direct").alias("pi_tags")).orderBy(F.desc("entities")).show()
print("By level:")
df.groupBy("level").agg(F.count("*").alias("entities"), F.sum("pi_tag_count_direct").alias("pi_tags")).orderBy("level").show()
print("PI coverage:")
with_pi = df.filter(F.col("pi_tag_count_direct") > 0).count()
print(f"  Entities with PI mapping: {with_pi} / {df.count()}")
print("\nSample RV3 with PI:")
df.filter((F.col("plant") == "RV3") & (F.col("pi_tag_count_direct") > 0)).select("equipment_name", "pi_prefixes_direct", "pi_tag_count_direct", "level").orderBy(F.desc("pi_tag_count_direct")).show(10, truncate=60)

print("=" * 60)
print("bridge_pi_tag_to_asset")
print("=" * 60)
df2 = spark.table("bridge_pi_tag_to_asset")
total = df2.count()
mapped = df2.filter(F.col("asset_id").isNotNull()).count()
print(f"Total tags: {total}")
print(f"Mapped to AIMM: {mapped} ({round(mapped/total*100,1)}%)")
print(f"Unmapped: {total - mapped}")
print("\nBy role:")
df2.groupBy("tag_role", "downtime_relevance").agg(F.count("*").alias("tags")).orderBy(F.desc("tags")).show()
print("Sample mapped:")
df2.filter(F.col("asset_id").isNotNull()).select("Tag", "tag_description", "asset_id", "tag_role").show(10, truncate=50)

In [ ]:
# === Promote dim_equipment and bridge_pi_tag_to_asset to gold schema ===

# Create gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Read from dbo
df_equip = spark.table("dim_equipment")
df_bridge = spark.table("bridge_pi_tag_to_asset")

# Write to gold schema (overwrites existing)
df_equip.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gold.dim_equipment")
df_bridge.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("gold.bridge_pi_tag_to_asset")

print(f"gold.dim_equipment: {df_equip.count()} rows")
print(f"gold.bridge_pi_tag_to_asset: {df_bridge.count()} rows")
print("\n=== Semantic Model Relationships ===")
print("""
  fact_pi[Tag]  *---1  bridge_pi_tag_to_asset[Tag]
  bridge_pi_tag_to_asset[asset_id]  *---1  dim_equipment[icare_id]

  Key columns:
    dim_equipment.icare_id = AIMM mtnoi (string, e.g. "7000520")
    bridge_pi_tag_to_asset.asset_id = same mtnoi (string)
    bridge_pi_tag_to_asset.Tag = PI tag name (e.g. "RV3:BFPP02A.AG")
    fact_pi.Tag = PI tag name (same)
""")

# Verify join integrity
orphan_check = df_bridge.filter(F.col("asset_id").isNotNull()).join(
    df_equip.select(F.col("icare_id")),
    df_bridge["asset_id"] == df_equip["icare_id"],
    "left_anti"
).count()
print(f"Orphan check (bridge asset_ids not in dim_equipment): {orphan_check}")
